# Phase 5 - Unified Evaluation Notebook

This merged notebook combines:

- XAI vs lesion-mask evaluation
- XAI vs pseudo-concept evaluation
- TIxAI / TAxAI metrics
- lesion-size stratified analysis
- melanoma vs non-melanoma analysis
- summary tables and plots

Source notebooks merged:
- currentMASTER
- mel_nonmel
- txai_size


# Phase 5 - XAI vs Pseudo-Concept Evaluation - {RUN_NAME}

This notebook evaluates whether XAI saliency maps align with the pseudo-concept maps generated from the lesion masks and images.

It expects the {RUN_NAME} outputs from the previous notebooks:

```text
data/<run_name>/<run_name>.csv

outputs/<run_name>/manifests/{RUN_NAME}_xai_manifest.csv
outputs/<run_name>/manifests/{RUN_NAME}_pseudo_concepts_manifest.csv

outputs/<run_name>/xai_maps/
outputs/<run_name>/pseudo_concepts/npz/
```

Expected metric rows:

```text
100 images × 3 XAI methods × 3 pseudo-concepts = 900 rows
```

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

## 1. Configuration

In [ ]:
# If the notebook is in src/, ROOT = Path("..").resolve() is usually correct.
# If this notebook is run from the repository root, change ROOT = Path(".").resolve().

ROOT = Path("..").resolve()

DATA_DIR = ROOT / "data"
OUTPUTS_DIR = ROOT / "outputs"

RUN_NAME = "pilot_1260_strat" # CHANGE THIS to match the {RUN_NAME} subset you are using. It should match the folder name in data/ and outputs/.

RUN_CSV = DATA_DIR / RUN_NAME / f"{RUN_NAME}.csv"

XAI_MANIFEST = OUTPUTS_DIR / RUN_NAME / "manifests" / f"{RUN_NAME}_xai_manifest.csv"
PSEUDO_MANIFEST = OUTPUTS_DIR / RUN_NAME / "manifests" / f"{RUN_NAME}_pseudo_concepts_manifest.csv"

PHASE5_DIR = OUTPUTS_DIR / f"phase5_{RUN_NAME}"
METRICS_DIR = PHASE5_DIR / "metrics"
FIG_DIR = PHASE5_DIR / "figures"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

XAI_METHODS = ["gradcam", "lime", "shap"]

# These keys must match the keys saved inside the pseudo-concept .npz files.
CONCEPT_KEYS = {
    "asymmetry": "asymmetry",
    
    "border_default": "border_irregularity",
    "border_w8_dil2_sigma5": "border_w8_dil2_sigma5",
    "border_w12_dil4_sigma8": "border_w12_dil4_sigma8",
    "border_w16_dil6_sigma10": "border_w16_dil6_sigma10",
    "border_w16_dil6_sigma10_dist": "border_w16_dil6_sigma10_dist",

    "colour_heterogeneity": "colour_heterogeneity",
}

TOP_K_PERCENT = 20
EPS = 1e-8

print("ROOT:", ROOT)
print("RUN CSV:", RUN_CSV, "exists:", RUN_CSV.exists())
print("XAI manifest:", XAI_MANIFEST, "exists:", XAI_MANIFEST.exists())
print("Pseudo manifest:", PSEUDO_MANIFEST, "exists:", PSEUDO_MANIFEST.exists())
print("Phase 5 metrics dir:", METRICS_DIR, "exists:", METRICS_DIR.exists()) 
print("Phase 5 figures dir:", FIG_DIR, "exists:", FIG_DIR.exists())    


ROOT: /home/jessica/Projects/DCU
RUN CSV: /home/jessica/Projects/DCU/data/local_1260_260722/local_1260_260722.csv exists: False
XAI manifest: /home/jessica/Projects/DCU/outputs/local_1260_260722/manifests/local_1260_260722_xai_manifest.csv exists: False
Pseudo manifest: /home/jessica/Projects/DCU/outputs/local_1260_260722/manifests/local_1260_260722_pseudo_concepts_manifest.csv exists: False
Phase 5 metrics dir: /home/jessica/Projects/DCU/outputs/phase5_local_1260_260722/metrics exists: True
Phase 5 figures dir: /home/jessica/Projects/DCU/outputs/phase5_local_1260_260722/figures exists: True


## 2. Load and merge run manifests

In [3]:
def normalise_dataset_name(x):
    x = str(x).strip().lower()
    aliases = {
        "ham": "ham10000",
        "ham10000": "ham10000",
        "isic": "isic2018",
        "isic2018": "isic2018",
    }
    return aliases.get(x, x)


def ensure_exists(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


ensure_exists(RUN_CSV, "RUN CSV")
ensure_exists(XAI_MANIFEST, "XAI manifest")
ensure_exists(PSEUDO_MANIFEST, "Pseudo-concept manifest")

run_csv = pd.read_csv(RUN_CSV)
xai_manifest = pd.read_csv(XAI_MANIFEST)
pseudo_manifest = pd.read_csv(PSEUDO_MANIFEST)

for name, df in [
    ("run", run_csv),
    ("xai_manifest", xai_manifest),
    ("pseudo_manifest", pseudo_manifest),
]:
    if "dataset" not in df.columns or "stem" not in df.columns:
        raise ValueError(f"{name} must contain columns: dataset, stem")

    df["dataset"] = df["dataset"].map(normalise_dataset_name)
    df["stem"] = df["stem"].astype(str)

print("RUN rows:", len(run_csv))
print("XAI manifest rows:", len(xai_manifest))
print("Pseudo manifest rows:", len(pseudo_manifest))

display(run_csv.head())
display(xai_manifest.head())
display(pseudo_manifest.head())

FileNotFoundError: RUN CSV not found: /home/jessica/Projects/DCU/data/local_1260_260722/local_1260_260722.csv

In [ ]:
run_resolved = (
    run_csv
    .merge(
        xai_manifest,
        on=["dataset", "stem"],
        how="left",
        suffixes=("", "_xai"),
    )
    .merge(
        pseudo_manifest,
        on=["dataset", "stem"],
        how="left",
        suffixes=("", "_pseudo"),
    )
)

print("Merged run rows:", len(run_resolved))
display(run_resolved.head())

for method in XAI_METHODS:
    col = f"xai_{method}_path"
    if col in run_resolved.columns:
        print(f"Rows with {col}:", run_resolved[col].notna().sum())
    else:
        print(f"Missing column: {col}")

if "pseudo_npz_path" in run_resolved.columns:
    print("Rows with pseudo_npz_path:", run_resolved["pseudo_npz_path"].notna().sum())
else:
    print("Missing column: pseudo_npz_path")

## 3. Validate file paths

In [ ]:
def resolve_path(p):
    if p is None or pd.isna(p):
        return None

    p = Path(str(p))

    if p.is_absolute():
        return p

    # First try relative to current notebook working directory.
    if p.exists():
        return p.resolve()

    # Then try relative to repository root.
    p2 = ROOT / p
    if p2.exists():
        return p2.resolve()

    return p


path_check_records = []

for _, row in run_resolved.iterrows():
    rec = {
        "dataset": row["dataset"],
        "stem": row["stem"],
    }

    for method in XAI_METHODS:
        col = f"xai_{method}_path"
        p = resolve_path(row.get(col)) if col in run_resolved.columns else None
        rec[f"{method}_exists"] = bool(p is not None and Path(p).exists())

    p = resolve_path(row.get("pseudo_npz_path")) if "pseudo_npz_path" in run_resolved.columns else None
    rec["pseudo_exists"] = bool(p is not None and Path(p).exists())

    path_check_records.append(rec)

path_check = pd.DataFrame(path_check_records)

display(path_check.head())

print("File availability summary:")
display(path_check.drop(columns=["dataset", "stem"]).sum().to_frame("count"))

missing_any = path_check[
    ~path_check[[f"{m}_exists" for m in XAI_METHODS] + ["pseudo_exists"]].all(axis=1)
]

print("Rows missing at least one required file:", len(missing_any))
display(missing_any.head(20))

In [ ]:
# Optional quick check: inspect the first pseudo-concept NPZ file.
first_pseudo_raw = run_resolved["pseudo_npz_path"].dropna().iloc[0]
first_pseudo = resolve_path(first_pseudo_raw)

with np.load(first_pseudo, allow_pickle=False) as data:
    print("First pseudo-concept file:", first_pseudo)
    print("Available keys:", data.files)


## 4. Map loading and metric helpers

In [ ]:
# min-max normalize to [0,1], handle common channel-first/last cases, and ensure 2D output.
def normalize_map(x):
    arr = np.asarray(x)
    arr = np.squeeze(arr)

    # Convert common channel-first or channel-last attribution arrays to 2D.
    if arr.ndim == 3:
        if arr.shape[0] in [1, 3, 4]:
            arr = np.mean(np.abs(arr), axis=0)
        elif arr.shape[-1] in [1, 3, 4]:
            arr = np.mean(np.abs(arr), axis=-1)

    if arr.ndim != 2:
        raise ValueError(f"Expected 2D map after conversion, got shape {arr.shape}")

    arr = arr.astype(np.float32)

    if not np.isfinite(arr).all():
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

    arr = arr - arr.min()
    denom = arr.max() + EPS
    arr = arr / denom

    return arr


def load_map_file(path, key=None):
    path = resolve_path(path)
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    suffix = path.suffix.lower()

    if suffix == ".npy":
        return normalize_map(np.load(path, allow_pickle=False))

    if suffix == ".npz":
        data = np.load(path, allow_pickle=False)

        if key is not None:
            if key not in data.files:
                raise KeyError(
                    f"Key '{key}' not found in {path}. Available keys: {data.files}"
                )
            return normalize_map(data[key])

        return normalize_map(data[data.files[0]])

    img = Image.open(path).convert("L")
    return normalize_map(np.asarray(img))


def topk_binary(x, top_k_percent=20):
    x = normalize_map(x)
    threshold = np.percentile(x, 100 - top_k_percent)
    return x >= threshold


def binary_from_map(x, threshold=0.5):
    x = normalize_map(x)
    return x >= threshold

# IOU
def iou_score(a, b):
    a = np.asarray(a).astype(bool)
    b = np.asarray(b).astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter / (union + EPS))

# DICE
def dice_score(a, b):
    a = np.asarray(a).astype(bool)
    b = np.asarray(b).astype(bool)
    inter = np.logical_and(a, b).sum()
    return float((2 * inter) / (a.sum() + b.sum() + EPS))

# SIR
def saliency_inside_ratio(saliency, region):
    saliency = normalize_map(saliency)
    region = np.asarray(region).astype(bool)

    total = saliency.sum()
    if total <= EPS:
        return 0.0

    return float(saliency[region].sum() / (total + EPS))


def mean_inside_outside_ratio(saliency, region):
    saliency = normalize_map(saliency)
    region = np.asarray(region).astype(bool)

    if region.sum() == 0:
        return 0.0

    inside = saliency[region].mean()

    if (~region).sum() == 0:
        return np.nan

    outside = saliency[~region].mean()

    return float(inside / (outside + EPS))


def pearson_corr(a, b):
    a = normalize_map(a).ravel()
    b = normalize_map(b).ravel()

    if a.std() <= EPS or b.std() <= EPS:
        return 0.0

    return float(np.corrcoef(a, b)[0, 1])


# TIxAI
def tixai_score(
    saliency: np.ndarray,
    region: np.ndarray,
    eps: float = EPS,
) -> float:
    """
    Compute TIxAI trustworthiness metric.
    The saliency map is normalised before computation.
    In the main pipeline, maps are already normalised by load_map_file(),
    so this second normalisation is normally redundant but harmless.

    Parameters
    ----------
    saliency : np.ndarray
        Continuous saliency map in [0,1].

    region : np.ndarray
        Binary region mask.

    Returns
    -------
    float
        TIxAI score in [0,1].
    """
    saliency = normalize_map(saliency).astype(np.float32)
    mask = np.asarray(region).astype(bool)

    inside = saliency[mask]
    outside = saliency[~mask]

    inside_relevance = inside.mean() if inside.size > 0 else 0.0
    outside_relevance = outside.mean() if outside.size > 0 else 0.0

    tixai = inside_relevance / (
        inside_relevance + outside_relevance + eps
    )

    return float(tixai)

## 5. Inspect pseudo-concept NPZ keys

In [ ]:
if "pseudo_npz_path" not in run_resolved.columns:
    raise ValueError("pseudo_npz_path column is missing from the merged dataframe.")

first_pseudo = run_resolved["pseudo_npz_path"].dropna().iloc[0]
first_pseudo = resolve_path(first_pseudo)

print("First pseudo-concept file:", first_pseudo)

with np.load(first_pseudo, allow_pickle=False) as data:
    print("Available keys:", data.files)

print("Expected keys:")
for concept_name, key in CONCEPT_KEYS.items():
    print(f"{concept_name}: {key}")

## 6. Compute XAI vs lesion mask metrics


In [ ]:
# ============================================================
# XAI vs lesion mask evaluation

mask_records = []

for _, row in tqdm(run_resolved.iterrows(), total=len(run_resolved)):

    pseudo_path = row.get("pseudo_npz_path")

    if pseudo_path is None or pd.isna(pseudo_path):
        continue

    try:
        pseudo_path = resolve_path(pseudo_path)

        with np.load(pseudo_path, allow_pickle=False) as data:
            mask = data["mask"].astype(np.float32)

    except Exception as e:
        print("Could not load mask from pseudo NPZ:", pseudo_path, e)
        continue

    mask_bin = binary_from_map(mask, threshold=0.5)

    for method in XAI_METHODS:
        xai_path = row.get(f"xai_{method}_path")

        if xai_path is None or pd.isna(xai_path):
            continue

        try:
            xai_map = load_map_file(resolve_path(xai_path))
        except Exception as e:
            print("Could not load XAI:", xai_path, e)
            continue

        xai_bin = topk_binary(xai_map, TOP_K_PERCENT)

        mask_records.append({
            "dataset": row["dataset"],
            "stem": row["stem"],
            "xai_method": method,
            "iou": iou_score(xai_bin, mask_bin),
            "dice": dice_score(xai_bin, mask_bin),
            "sir": saliency_inside_ratio(xai_map, mask_bin),
            "tixai": tixai_score(xai_map, mask_bin),
            "inside_outside_ratio": mean_inside_outside_ratio(xai_map, mask_bin),
            "mask_area_ratio": mask_bin.mean(),
            "xai_area_ratio": xai_bin.mean(),
        })

mask_metrics = pd.DataFrame(mask_records)

print("Mask evaluation rows:", len(mask_metrics))
display(mask_metrics.head())
tableIII = mask_metrics.groupby("xai_method")[["iou", "dice", "sir", "tixai", "xai_area_ratio"]].mean().reset_index()


In [ ]:
# Optional diagnostics after mask evaluation.
print(tableIII)
print("run_resolved shape:", run_resolved.shape)
print("XAI path columns:", [c for c in run_resolved.columns if c.startswith("xai_") and c.endswith("_path")])
print("Pseudo rows:", run_resolved["pseudo_npz_path"].notna().sum())
print("Mask metric rows:", len(mask_metrics))


## 7. Compute XAI vs pseudo-concept metrics

In [ ]:
records = []
load_errors = []

for _, row in tqdm(run_resolved.iterrows(), total=len(run_resolved)):
    dataset = row["dataset"]
    stem = str(row["stem"])

    pseudo_path = row.get("pseudo_npz_path")
    if pseudo_path is None or pd.isna(pseudo_path):
        continue

    pseudo_path = resolve_path(pseudo_path)

    for method in XAI_METHODS:
        xai_col = f"xai_{method}_path"

        if xai_col not in run_resolved.columns:
            continue

        xai_path = row.get(xai_col)

        if xai_path is None or pd.isna(xai_path):
            continue

        xai_path = resolve_path(xai_path)

        try:
            xai_map = load_map_file(xai_path)
            xai_bin = topk_binary(xai_map, TOP_K_PERCENT)
        except Exception as e:
            load_errors.append({
                "dataset": dataset,
                "stem": stem,
                "type": "xai",
                "method": method,
                "path": str(xai_path),
                "error": str(e),
            })
            continue

        for concept_name, concept_key in CONCEPT_KEYS.items():
            try:
                concept_map = load_map_file(pseudo_path, key=concept_key)
                concept_bin = binary_from_map(concept_map, threshold=0.5)
            except Exception as e:
                load_errors.append({
                    "dataset": dataset,
                    "stem": stem,
                    "type": "concept",
                    "concept": concept_name,
                    "key": concept_key,
                    "path": str(pseudo_path),
                    "error": str(e),
                })
                continue

            records.append({
                "dataset": dataset,
                "stem": stem,
                "xai_method": method,
                "concept": concept_name,
                "top_k_percent": TOP_K_PERCENT,
                "iou": iou_score(xai_bin, concept_bin),
                "dice": dice_score(xai_bin, concept_bin),
                "sir": saliency_inside_ratio(xai_map, concept_bin),
                "tixai": tixai_score(xai_map, concept_bin),
                "inside_outside_ratio": mean_inside_outside_ratio(xai_map, concept_bin),
                "pearson_corr": pearson_corr(xai_map, concept_map),
                "concept_area_ratio": float(concept_bin.mean()),
                "xai_area_ratio": float(xai_bin.mean()),
                "xai_path": str(xai_path),
                "pseudo_npz_path": str(pseudo_path),
            })

metrics = pd.DataFrame(records)
errors = pd.DataFrame(load_errors)

metrics_path = METRICS_DIR / f"phase5_{RUN_NAME}_xai_concept_metrics.csv"
errors_path = METRICS_DIR / f"phase5_{RUN_NAME}_load_errors.csv"

metrics.to_csv(metrics_path, index=False)
errors.to_csv(errors_path, index=False)

print("Metric rows:", len(metrics))
print("Expected if complete:", len(run_resolved) * len(XAI_METHODS) * len(CONCEPT_KEYS))
print("Saved metrics:", metrics_path)

print("Load errors:", len(errors))
print("Saved errors:", errors_path)

display(metrics.head())
display(errors.head())

# Generate a summary of the pseudo-concept sizes, to be used in the evaluation of XAI vs pseudo-concept metrics.

In [ ]:
# ============================================================
# Pseudo-concept size summary
# One row per image/concept, because concept_area_ratio is independent of the XAI method.

concept_size = (
    metrics
    .drop_duplicates(subset=["stem", "concept"])
    .groupby("concept")
    .agg(
        n=("stem", "count"),
        mean_concept_area_ratio=("concept_area_ratio", "mean"),
        median_concept_area_ratio=("concept_area_ratio", "median"),
        std_concept_area_ratio=("concept_area_ratio", "std"),
    )
    .reset_index()
)

concept_size["mean_percent"] = concept_size["mean_concept_area_ratio"] * 100
concept_size["median_percent"] = concept_size["median_concept_area_ratio"] * 100
concept_size["std_percent"] = concept_size["std_concept_area_ratio"] * 100

display(concept_size.sort_values("mean_percent", ascending=False))

# For reporting, we only include the concepts that are used in the main evaluation.
report_concept_size = concept_size[
    concept_size["concept"].isin([
        "asymmetry",
        "border_w16_dil6_sigma10",
        "colour_heterogeneity",
    ])
][["concept", "mean_percent"]]

display(report_concept_size)

## 8. Summary tables

In [ ]:

# ============================================================
# XAI vs lesion mask summary

if len(mask_metrics) == 0:
    raise ValueError("No mask metrics were generated.")

mask_summary = (
    mask_metrics
    .groupby(["xai_method"])
    .agg(
        n=("stem", "count"),
        mean_iou=("iou", "mean"),
        mean_dice=("dice", "mean"),
        mean_sir=("sir", "mean"),
        mean_tixai=("tixai", "mean"),
        mean_inside_outside_ratio=("inside_outside_ratio", "mean"),
        mean_mask_area=("mask_area_ratio", "mean"),
        mean_xai_area=("xai_area_ratio", "mean"),
    )
    .reset_index()
    .sort_values("mean_sir", ascending=False)
)

mask_summary_path = (
    METRICS_DIR
    / f"phase5_{RUN_NAME}_mask_summary_by_method.csv"
)

mask_summary.to_csv(mask_summary_path, index=False)

print("Saved mask summary:", mask_summary_path)

display(mask_summary)

In [ ]:
# ============================================================
# XAI vs pseudo-concept summary


if len(metrics) == 0:
    raise ValueError("No pseudo-concept metrics were generated.")

summary = (
    metrics
    .groupby(["xai_method", "concept"])
    .agg(
        n=("stem", "count"),
        mean_iou=("iou", "mean"),
        mean_dice=("dice", "mean"),
        mean_sir=("sir", "mean"),
        mean_tixai=("tixai", "mean"),
        mean_inside_outside_ratio=("inside_outside_ratio", "mean"),
        mean_pearson_corr=("pearson_corr", "mean"),
        mean_concept_area=("concept_area_ratio", "mean"),
    )
    .reset_index()
    .sort_values(["concept", "mean_sir"], ascending=[True, False])
)

summary_path = METRICS_DIR / f"phase5_{RUN_NAME}_summary_by_method_concept.csv"
summary.to_csv(summary_path, index=False)

print("Saved pseudo-concept summary:", summary_path)
display(summary)



dataset_summary = (
    metrics
    .groupby(["dataset", "xai_method", "concept"])
    .agg(
        n=("stem", "count"),
        mean_iou=("iou", "mean"),
        mean_dice=("dice", "mean"),
        mean_sir=("sir", "mean"),
        mean_tixai=("tixai", "mean"),
        mean_pearson_corr=("pearson_corr", "mean"),
    )
    .reset_index()
    .sort_values(["dataset", "concept", "mean_sir"], ascending=[True, True, False])
)

dataset_summary_path = METRICS_DIR / f"phase5_{RUN_NAME}_summary_by_dataset.csv"
dataset_summary.to_csv(dataset_summary_path, index=False)

print("Saved dataset summary:", dataset_summary_path)
display(dataset_summary)

In [ ]:
# ============================================================
# Pseudo-concept performance by lesion size class


# set metric used
metric_used = "dice"  # Change this to the desired metric column name, e.g., "iou", "dice", "sir", "tixai", etc.


size_summary = (
    metrics
    .merge(
        run_resolved[["dataset", "stem", "mask_size_class"]],
        on=["dataset", "stem"],
        how="left"
    )
    .groupby(["mask_size_class", "xai_method", "concept"])
    .agg(
        n=("stem", "count"),
        mean_iou=("iou", "mean"),
        mean_dice=("dice", "mean"),
        mean_sir=("sir", "mean"),
        mean_tixai=("tixai", "mean"),
        mean_pearson_corr=("pearson_corr", "mean"),
        mean_concept_area=("concept_area_ratio", "mean"),
    )
    .reset_index()
    .sort_values(["concept", "mask_size_class", f"mean_{metric_used}"], ascending=[True, True, False])
)

size_summary_path = METRICS_DIR / f"phase5_{RUN_NAME}_summary_by_size_class.csv"
size_summary.to_csv(size_summary_path, index=False)

display(size_summary)
print("Saved:", size_summary_path)




normal_summary = (
    size_summary[size_summary["mask_size_class"] == "normal"]
    .sort_values(["concept", f"mean_{metric_used}"], ascending=[True, False])
)

display(normal_summary)


for concept in ["asymmetry", "border_w16_dil6_sigma10"]:
    subset = size_summary[size_summary["concept"] == concept]

    pivot = subset.pivot_table(
        index="mask_size_class",
        columns="xai_method",
        values=f"mean_{metric_used}",
        aggfunc="mean"
    )

    ax = pivot.plot(kind="bar", figsize=(8, 4))
    ax.set_title(f"Pseudo-concept {metric_used} by lesion size - {concept}")
    ax.set_ylabel(f"mean_{metric_used}")
    ax.set_xlabel("mask_size_class")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 9. Label-Stratified Analysis (Melanoma vs Non-Melanoma)

### 9.1 Pseudo-Concept Prevalence Analysis

In [ ]:
# ============================================================
# Pseudo-Concept Prevalence by Diagnostic Label

CORE_CONCEPTS = [
    "asymmetry",
    # "border_default",
    # "border_w8_dil2_sigma5",
    # "border_w12_dil4_sigma8",
    # "border_w16_dil6_sigma10",
    "border_w16_dil6_sigma10_dist",
    "colour_heterogeneity",
]

CONCEPT_LABELS = {
    "asymmetry": "Asymmetry",
    # "border_default": "Border default",
    # "border_w8_dil2_sigma5": "Border w8 d2 s5",
    # "border_w12_dil4_sigma8": "Border w12 d4 s8",
    # "border_w16_dil6_sigma10": "Border w16 d6 s10",
    "border_w16_dil6_sigma10_dist": "Border dist.",
    "colour_heterogeneity": "Colour heterogeneity",
}

ham_metrics = metrics[
    metrics["dataset"].str.lower().isin(["ham", "ham10000"])
].copy()

# Keep only the selected pseudo-concepts
ham_metrics = ham_metrics[
    ham_metrics["concept"].isin(CORE_CONCEPTS)
].copy()

label_info = run_resolved[
    ["dataset", "stem", "label_name"]
].drop_duplicates()

ham_metrics = ham_metrics.merge(
    label_info,
    on=["dataset", "stem"],
    how="left"
)

# concept_area_ratio is repeated for each XAI method
# keep one row per image/concept
concept_prevalence = (
    ham_metrics
    .drop_duplicates(subset=["stem", "concept"])
)

prevalence_summary = (
    concept_prevalence
    .groupby(["label_name", "concept"])
    .agg(
        n=("stem", "nunique"),
        mean_concept_area_ratio=("concept_area_ratio", "mean"),
        median_concept_area_ratio=("concept_area_ratio", "median"),
        std_concept_area_ratio=("concept_area_ratio", "std"),
    )
    .reset_index()
)

prevalence_summary["mean_percent"] = (
    prevalence_summary["mean_concept_area_ratio"] * 100
)

# Display labels
prevalence_summary["concept_label"] = (
    prevalence_summary["concept"].map(CONCEPT_LABELS)
)

# Enforce concept order
prevalence_summary["concept_label"] = pd.Categorical(
    prevalence_summary["concept_label"],
    categories=[CONCEPT_LABELS[c] for c in CORE_CONCEPTS],
    ordered=True,
)

prevalence_summary = prevalence_summary.sort_values(
    ["concept_label", "label_name"]
)

display(
    prevalence_summary[
        [
            "label_name",
            "concept_label",
            "n",
            "mean_percent",
            "median_concept_area_ratio",
            "std_concept_area_ratio",
        ]
    ].round(3)
)

In [ ]:
for metric_name in ["dice", "sir", "pearson_corr"]:
    print("\n", "=" * 80)
    print("Metric:", metric_name)
    print("Top 10")
    display(metrics.sort_values(metric_name, ascending=False).head(10))
    print("Bottom 10")
    display(metrics.sort_values(metric_name, ascending=True).head(10))

In [ ]:

plt.figure(figsize=(10,6))

sns.barplot(
    data=prevalence_summary,
    x="concept",
    y="mean_percent",
    hue="label_name"
)

plt.ylabel("% of Lesion Area")
plt.xlabel("Pseudo-Concept")
plt.title("Pseudo-Concept Prevalence by Diagnostic Label")

plt.xticks(rotation=30)
plt.tight_layout()

plt.savefig(
    FIG_DIR / "fig_pseudo_concept_prevalence_by_label.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### 9.2 XAI-to-Pseudo-Concept Alignment by Label

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# ------------------------------------------------------------
# HAM10000 diagnostic analysis: melanoma vs non-melanoma
# restricted to CORE_CONCEPTS only
# ------------------------------------------------------------

CORE_CONCEPTS = [
    "asymmetry",
    # "border_default",
    # "border_w8_dil2_sigma5",
    # "border_w12_dil4_sigma8",
    # "border_w16_dil6_sigma10",
    "border_w16_dil6_sigma10_dist",
    "colour_heterogeneity",
]

CONCEPT_LABELS = {
    "asymmetry": "Asymmetry",
    # "border_default": "Border default",
    # "border_w8_dil2_sigma5": "Border w8 d2 s5",
    # "border_w12_dil4_sigma8": "Border w12 d4 s8",
    # "border_w16_dil6_sigma10": "Border w16 d6 s10",
    "border_w16_dil6_sigma10_dist": "Border dist.",
    "colour_heterogeneity": "Colour heterogeneity",
}


ham_metrics = metrics[
    metrics["dataset"].str.lower().isin(["ham", "ham10000"])
].copy()

# Keep only selected pseudo-concepts
ham_metrics = ham_metrics[ham_metrics["concept"].isin(CORE_CONCEPTS)].copy()

# Add readable concept labels
ham_metrics["concept_label"] = ham_metrics["concept"].map(CONCEPT_LABELS)

label_info = run_resolved[
    ["dataset", "stem", "label", "label_name"]
].drop_duplicates()

ham_metrics = ham_metrics.merge(
    label_info,
    on=["dataset", "stem"],
    how="left"
)

summary = (
    ham_metrics
    .groupby(["label_name", "xai_method", "concept", "concept_label"])
    .agg(
        n=("stem", "nunique"),
        mean_iou=("iou", "mean"),
        mean_dice=("dice", "mean"),
        mean_sir=("sir", "mean"),
        mean_pearson_corr=("pearson_corr", "mean"),
    )
    .reset_index()
)

# Optional: enforce display order
summary["concept_label"] = pd.Categorical(
    summary["concept_label"],
    categories=[CONCEPT_LABELS[c] for c in CORE_CONCEPTS],
    ordered=True,
)

summary = summary.sort_values(["label_name", "concept_label", "xai_method"])

# display(summary)

metrics_to_test = ["iou", "dice", "sir", "tixai"]
tests = []

for method in ham_metrics["xai_method"].dropna().unique():
    for concept in CORE_CONCEPTS:
        subset = ham_metrics[
            (ham_metrics["xai_method"] == method) &
            (ham_metrics["concept"] == concept)
        ]

        if subset.empty:
            continue

        for metric in metrics_to_test:
            if metric not in subset.columns:
                continue

            mel = subset[subset["label"] == 1][metric].dropna()
            nonmel = subset[subset["label"] == 0][metric].dropna()

            if len(mel) >= 5 and len(nonmel) >= 5:
                stat, p = mannwhitneyu(
                    mel,
                    nonmel,
                    alternative="two-sided"
                )

                n1, n2 = len(mel), len(nonmel)
                rank_biserial = (2 * stat / (n1 * n2)) - 1

                tests.append({
                    "xai_method": method,
                    "concept": concept,
                    "concept_label": CONCEPT_LABELS[concept],
                    "metric": metric,
                    "n_melanoma": n1,
                    "n_nonmelanoma": n2,
                    "mean_melanoma": mel.mean(),
                    "mean_nonmelanoma": nonmel.mean(),
                    "median_melanoma": mel.median(),
                    "median_nonmelanoma": nonmel.median(),
                    "difference_mean": mel.mean() - nonmel.mean(),
                    "difference_median": mel.median() - nonmel.median(),
                    "mannwhitney_u": stat,
                    "rank_biserial_effect": rank_biserial,
                    "p_value": p,
                })

label_tests = pd.DataFrame(tests)

if not label_tests.empty:
    label_tests["p_fdr"] = multipletests(
        label_tests["p_value"],
        method="fdr_bh"
    )[1]

    label_tests["concept_label"] = pd.Categorical(
        label_tests["concept_label"],
        categories=[CONCEPT_LABELS[c] for c in CORE_CONCEPTS],
        ordered=True,
    )

    label_tests = label_tests.sort_values(
        ["concept_label", "xai_method", "metric"]
    )
else:
    label_tests["p_fdr"] = np.nan

interesting = label_tests[
    (label_tests["p_value"] < 0.10) &
    (label_tests["rank_biserial_effect"].abs() > 0.20)
].copy()

display(
    interesting.sort_values(
        ["p_value", "rank_biserial_effect"],
        ascending=[True, False]
    )
)

display(
    label_tests.sort_values(
        "rank_biserial_effect",
        key=lambda s: s.abs(),
        ascending=False
    ).head(20)
)

### 9.3 Lesion Size as a Potential Confounder

In [ ]:
### Are melanoma lesions smaller than non-melanoma lesions in HAM10000, and could this explain any differences in XAI performance?

size_info = run_resolved[
    ["dataset", "stem", "mask_pixels_224", "mask_size_class"]
].drop_duplicates()

ham_metrics = ham_metrics.merge(
    size_info,
    on=["dataset", "stem"],
    how="left"
)

ham_images = ham_metrics.drop_duplicates(subset=["dataset", "stem"])

display(
    ham_images
    .groupby("label_name")["mask_pixels_224"]
    .agg(["count", "mean", "median", "std", "min", "max"])
)
from scipy.stats import mannwhitneyu

mel_size = ham_images[ham_images["label"] == 1]["mask_pixels_224"].dropna()
nonmel_size = ham_images[ham_images["label"] == 0]["mask_pixels_224"].dropna()

stat, p = mannwhitneyu(mel_size, nonmel_size, alternative="two-sided")

print("Melanoma mean size:", mel_size.mean())
print("Non-melanoma mean size:", nonmel_size.mean())
print("p-value:", p)

plt.figure(figsize=(7, 5))

sns.boxplot(
    data=ham_images,
    x="label_name",
    y="mask_pixels_224"
)

plt.title("Lesion Size by Class (HAM, 224x224 mask pixels)")
plt.tight_layout()
plt.show()
plt.savefig(
    FIG_DIR / "fig_lesion_size_by_label.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

print(ham_metrics["xai_method"].unique())
print(ham_metrics["concept"].unique())
subset = ham_metrics[
    (ham_metrics["xai_method"] == "shap") &
    (ham_metrics["concept"] == "colour_heterogeneity")
]

plt.figure(figsize=(8,5))

sns.boxplot(
    data=subset,
    x="label_name",
    y="pearson_corr"
)

plt.title("SHAP vs Colour Heterogeneity Correlation")
plt.show()

## 9. Best and worst examples

In [ ]:
for metric_name in ["dice", "sir", "pearson_corr"]:
    print("\n", "=" * 80)
    print("Metric:", metric_name)
    print("Top 10")
    display(metrics.sort_values(metric_name, ascending=False).head(10))
    print("Bottom 10")
    display(metrics.sort_values(metric_name, ascending=True).head(10))

## 10. Bar plots

This older bar-plot cell was replaced by the two separate plotting cells below: pseudo-concepts and lesion masks.


In [ ]:
# ============================================================
# Bar plots: XAI vs pseudo-concepts only

plot_metrics = [
    "mean_dice",
    "mean_sir",
    "mean_iou",
    "mean_tixai",
]

if "mean_pearson_corr" in summary.columns:
    plot_metrics.append("mean_pearson_corr")

for metric_name in plot_metrics:
    if metric_name not in summary.columns:
        print(f"Skipping {metric_name}: not found in summary")
        continue

    pivot = summary.pivot_table(
        index="concept",
        columns="xai_method",
        values=metric_name,
        aggfunc="mean"
    )

    ax = pivot.plot(kind="bar", figsize=(10, 5))
    ax.set_title(f"{RUN_NAME} - pseudo-concepts - {metric_name}")
    ax.set_ylabel(metric_name)
    ax.set_xlabel("Pseudo-concept")
    ax.legend(title="XAI method")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

    fig_path = FIG_DIR / f"phase5_{RUN_NAME}_pseudo_{metric_name}.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()

    print("Saved:", fig_path)

In [ ]:
# ============================================================
# Bar plots: XAI vs lesion mask only
# ============================================================

mask_plot_metrics = [
    "mean_iou",
    "mean_dice",
    "mean_sir",
    "mean_tixai",
]

for metric_name in mask_plot_metrics:
    if metric_name not in mask_summary.columns:
        print(f"Skipping {metric_name}: not found in mask_summary")
        continue

    ax = mask_summary.plot(
        x="xai_method",
        y=metric_name,
        kind="bar",
        legend=False,
        figsize=(7, 4)
    )

    ax.set_title(f"{RUN_NAME} - lesion mask - {metric_name}")
    ax.set_ylabel(metric_name)
    ax.set_xlabel("XAI method")
    plt.xticks(rotation=0)
    plt.tight_layout()

    fig_path = FIG_DIR / f"phase5_{RUN_NAME}_mask_{metric_name}.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()

    print("Saved:", fig_path)

## 11. Visual sanity-check grids

In [ ]:
def find_image_path_from_row(row):
    # Try common path columns from the original run/preprocessing manifests.
    candidates = [
        "image_path",
        "img_path",
        "resized_image_path",
        "preprocessed_image_path",
        "path",
    ]

    for col in candidates:
        if col in row.index and pd.notna(row[col]):
            p = resolve_path(row[col])
            if p is not None and Path(p).exists():
                return Path(p)

    return None


def load_rgb_image(path):
    return np.asarray(Image.open(path).convert("RGB"))


def show_or_blank(ax, arr, title="", cmap=None, alpha=1.0):
    ax.set_title(title, fontsize=8)
    ax.axis("off")

    if arr is None:
        ax.text(0.5, 0.5, "missing", ha="center", va="center")
        return

    ax.imshow(arr, cmap=cmap, alpha=alpha, vmin=0 if cmap else None, vmax=1 if cmap else None)


def make_visual_grid(rows, method="gradcam", concept_keys=CONCEPT_KEYS, max_images=8, save_path=None):
    rows = rows.head(max_images)
    n = len(rows)

    if n == 0:
        print("No rows to plot.")
        return

    ncols = 2 + len(concept_keys)

    fig, axes = plt.subplots(n, ncols, figsize=(3.2 * ncols, 3.2 * n))

    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_i, (_, row) in enumerate(rows.iterrows()):
        stem = str(row["stem"])
        dataset = str(row["dataset"])

        image_path = find_image_path_from_row(row)
        image = load_rgb_image(image_path) if image_path is not None else None

        xai = None
        xai_col = f"xai_{method}_path"
        if xai_col in row.index and pd.notna(row[xai_col]):
            try:
                xai = load_map_file(row[xai_col])
            except Exception as e:
                print(f"Could not load XAI for {dataset}/{stem}: {e}")

        pseudo_path = row.get("pseudo_npz_path")

        show_or_blank(axes[row_i, 0], image, f"{dataset}\n{stem}")

        if image is not None:
            axes[row_i, 1].imshow(image)
        show_or_blank(axes[row_i, 1], xai, method, cmap="hot", alpha=0.45)

        for concept_i, (concept_name, concept_key) in enumerate(concept_keys.items(), start=2):
            concept_map = None

            if pseudo_path is not None and pd.notna(pseudo_path):
                try:
                    concept_map = load_map_file(pseudo_path, key=concept_key)
                except Exception as e:
                    print(f"Could not load concept {concept_key} for {dataset}/{stem}: {e}")

            show_or_blank(
                axes[row_i, concept_i],
                concept_map,
                concept_name,
                cmap="viridis",
            )

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print("Saved:", save_path)

    plt.show()

In [ ]:
# Visualise a few examples for each XAI method.

example_rows = run_resolved.copy()

for method in XAI_METHODS:
    print("\n", "=" * 80)
    print("Visual grid:", method)

    save_path = FIG_DIR / f"phase5_{RUN_NAME}_visual_grid_{method}.png"

    make_visual_grid(
        example_rows,
        method=method,
        max_images=8,
        save_path=save_path,
    )

## for report: rewritten version. It keeps only one border concept and shows Original + 3 XAI methods + selected pseudo-concepts in the same grid

In [ ]:
# Visual grid: Original | Mask | Grad-CAM | SHAP | LIME | selected pseudo-concepts

SELECTED_CONCEPT_KEYS = {
    "Asymmetry": "asymmetry",
    "Border irregularity": "border_w16_dil6_sigma10_dist",
    "Colour heterogeneity": "colour_heterogeneity",
}

XAI_METHODS_TO_SHOW = ["gradcam", "shap", "lime"]


def normalize_map(arr):
    arr = np.asarray(arr).astype(float)

    if arr.ndim == 3:
        arr = arr[..., 0]

    arr_min = np.nanmin(arr)
    arr_max = np.nanmax(arr)

    if not np.isfinite(arr_min) or not np.isfinite(arr_max) or arr_max <= arr_min:
        return np.zeros_like(arr, dtype=float)

    return (arr - arr_min) / (arr_max - arr_min)


def resize_map_to_image(map_arr, image):
    if map_arr is None or image is None:
        return map_arr

    target_h, target_w = image.shape[:2]

    if map_arr.shape[:2] == (target_h, target_w):
        return normalize_map(map_arr)

    map_norm = normalize_map(map_arr)
    map_img = Image.fromarray((map_norm * 255).astype(np.uint8))
    map_img = map_img.resize((target_w, target_h), resample=Image.BILINEAR)

    return np.asarray(map_img).astype(float) / 255.0


def get_mask_for_row(row):
    if "pseudo_npz_path" not in row.index or pd.isna(row["pseudo_npz_path"]):
        return None

    pseudo_path = resolve_path(row["pseudo_npz_path"])

    if pseudo_path is None or not Path(pseudo_path).exists():
        return None

    data = np.load(pseudo_path)

    if "mask" not in data.files:
        print("No mask key found. Available keys:", data.files)
        return None

    return (data["mask"] > 0).astype(float)


def show_or_blank(ax, arr, title="", cmap=None, alpha=1.0):
    ax.set_title(title, fontsize=8)
    ax.axis("off")

    if arr is None:
        ax.text(0.5, 0.5, "missing", ha="center", va="center")
        return

    if cmap is None:
        ax.imshow(arr)
    else:
        ax.imshow(arr, cmap=cmap, alpha=alpha, vmin=0, vmax=1)


def draw_original_with_label(ax, image, dataset, stem):
    ax.axis("off")

    if image is not None:
        ax.imshow(image)
    else:
        ax.text(0.5, 0.5, "missing", ha="center", va="center")

    ax.text(
        0.02,
        0.98,
        f"{dataset}\n{stem}",
        transform=ax.transAxes,
        fontsize=8,
        fontweight="bold",
        va="top",
        ha="left",
        color="black",
        bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=2),
    )


def make_visual_grid_all_xai(
    rows,
    xai_methods=XAI_METHODS_TO_SHOW,
    concept_keys=SELECTED_CONCEPT_KEYS,
    max_images=3,
    random_state=5,
    save_path=None,
):
    rows = rows.copy()

    if len(rows) > max_images:
        rows = rows.sample(n=max_images, random_state=random_state)

    rows = rows.reset_index(drop=True)
    n = len(rows)

    if n == 0:
        print("No rows to plot.")
        return

    ncols = 2 + len(xai_methods) + len(concept_keys)

    fig, axes = plt.subplots(
        n,
        ncols,
        figsize=(3.2 * ncols, 3.2 * n),
        squeeze=False,
    )

    col_titles = (
        ["Original", "Mask"]
        + [m.upper() if m != "gradcam" else "Grad-CAM" for m in xai_methods]
        + list(concept_keys.keys())
    )

    for row_i, (_, row) in enumerate(rows.iterrows()):
        stem = str(row["stem"])
        dataset = str(row["dataset"])

        image_path = find_image_path_from_row(row)
        image = load_rgb_image(image_path) if image_path is not None else None

        # Column titles only on first row
        if row_i == 0:
            for col_i, title in enumerate(col_titles):
                axes[row_i, col_i].set_title(
                    title,
                    fontsize=9,
                    fontweight="bold",
                )

        # Original
        draw_original_with_label(
            axes[row_i, 0],
            image,
            dataset,
            stem,
        )

        # Mask
        mask = get_mask_for_row(row)
        mask = resize_map_to_image(mask, image)

        show_or_blank(
            axes[row_i, 1],
            mask,
            title="",
            cmap="gray",
        )

        # XAI methods
        xai_start_col = 2

        for method_i, method in enumerate(xai_methods):
            xai = None
            xai_col = f"xai_{method}_path"

            if xai_col in row.index and pd.notna(row[xai_col]):
                try:
                    xai = load_map_file(row[xai_col])
                    xai = resize_map_to_image(xai, image)
                except Exception as e:
                    print(f"Could not load {method} for {dataset}/{stem}: {e}")

            ax = axes[row_i, xai_start_col + method_i]
            ax.axis("off")

            if image is not None:
                ax.imshow(image)

            if xai is not None:
                ax.imshow(xai, cmap="hot", alpha=0.45, vmin=0, vmax=1)
            else:
                ax.text(0.5, 0.5, "missing", ha="center", va="center")

        # Pseudo-concepts
        pseudo_path = row.get("pseudo_npz_path")
        concept_start_col = 2 + len(xai_methods)

        for concept_i, (concept_name, concept_key) in enumerate(concept_keys.items()):
            concept_map = None

            if pseudo_path is not None and pd.notna(pseudo_path):
                try:
                    concept_map = load_map_file(pseudo_path, key=concept_key)
                    concept_map = resize_map_to_image(concept_map, image)
                except Exception as e:
                    print(
                        f"Could not load concept {concept_key} "
                        f"for {dataset}/{stem}: {e}"
                    )

            show_or_blank(
                axes[row_i, concept_start_col + concept_i],
                concept_map,
                title="",
                cmap="viridis",
            )

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print("Saved:", save_path)

    plt.show()


example_rows = run_resolved.copy()

save_path = FIG_DIR / f"phase5_{RUN_NAME}_visual_grid_all_xai_selected_concepts.png"

make_visual_grid_all_xai(
    example_rows,
    max_images=8,
    random_state=1,
    save_path=save_path,
)

## 12. Visual grids for one image, showing all XAI methods and pseudo-concepts

In [ ]:
# function to show a visual grid for a specific stem value, including the original image, mask, XAI methods, and selected pseudo-concepts.

SELECTED_CONCEPT_KEYS = {
    "Asymmetry": "asymmetry",
    "Border irregularity": "border_w16_dil6_sigma10_dist",
    "Colour heterogeneity": "colour_heterogeneity",
}

XAI_METHODS_TO_SHOW = ["gradcam", "shap", "lime"]

def show_visual_grid_for_stem(
    stem_value,
    rows=run_resolved,
    xai_methods=XAI_METHODS_TO_SHOW,
    concept_keys=SELECTED_CONCEPT_KEYS,
    save_path=None,
):
    matches = rows[rows["stem"].astype(str) == str(stem_value)].copy()

    if len(matches) == 0:
        print(f"No row found for stem: {stem_value}")
        return

    if len(matches) > 1:
        print(f"Multiple rows found for stem {stem_value}. Using the first one.")

    row = matches.iloc[0]

    stem = str(row["stem"])
    dataset = str(row["dataset"])

    image_path = find_image_path_from_row(row)
    image = load_rgb_image(image_path) if image_path is not None else None

    ncols = 2 + len(xai_methods) + len(concept_keys)

    fig, axes = plt.subplots(
        1,
        ncols,
        figsize=(3.2 * ncols, 3.2),
        squeeze=False,
    )

    axes = axes[0]

    col_titles = (
        ["Original", "Mask"]
        + [m.upper() if m != "gradcam" else "Grad-CAM" for m in xai_methods]
        + list(concept_keys.keys())
    )

    for col_i, title in enumerate(col_titles):
        axes[col_i].set_title(title, fontsize=9, fontweight="bold")
        axes[col_i].axis("off")

    # Original
    draw_original_with_label(
        axes[0],
        image,
        dataset,
        stem,
    )

    # Mask
    mask = get_mask_for_row(row)
    mask = resize_map_to_image(mask, image)

    show_or_blank(
        axes[1],
        mask,
        title="Mask",
        cmap="gray",
    )

    # XAI methods
    xai_start_col = 2

    for method_i, method in enumerate(xai_methods):
        xai = None
        xai_col = f"xai_{method}_path"

        if xai_col in row.index and pd.notna(row[xai_col]):
            try:
                xai = load_map_file(row[xai_col])
                xai = resize_map_to_image(xai, image)
            except Exception as e:
                print(f"Could not load {method} for {dataset}/{stem}: {e}")

        ax = axes[xai_start_col + method_i]
        ax.axis("off")

        if image is not None:
            ax.imshow(image)

        if xai is not None:
            ax.imshow(xai, cmap="hot", alpha=0.45, vmin=0, vmax=1)
        else:
            ax.text(0.5, 0.5, "missing", ha="center", va="center")

    # Pseudo-concepts
    pseudo_path = row.get("pseudo_npz_path")
    concept_start_col = 2 + len(xai_methods)

    for concept_i, (concept_name, concept_key) in enumerate(concept_keys.items()):
        concept_map = None

        if pseudo_path is not None and pd.notna(pseudo_path):
            try:
                concept_map = load_map_file(pseudo_path, key=concept_key)
                concept_map = resize_map_to_image(concept_map, image)
            except Exception as e:
                print(
                    f"Could not load concept {concept_key} "
                    f"for {dataset}/{stem}: {e}"
                )

        show_or_blank(
            axes[concept_start_col + concept_i],
            concept_map,
            title=concept_name,
            cmap="viridis",
        )

    plt.suptitle(
        f"{dataset} / {stem}",
        fontsize=11,
        fontweight="bold",
        y=1.05,
    )

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print("Saved:", save_path)

    plt.show()


    # Example usage:
image_stem = "ISIC_0028162"  # Replace with the desired stem value

# xai_methods_to_show = ["gradcam", "shap", "lime"]  # Specify the XAI methods to include
xai_methods_to_show = []  # Specify the XAI methods to include


concept_keys_to_show = {"Asymmetry": "asymmetry", "Border irregularity": "border_w16_dil6_sigma10_dist", "Colour heterogeneity": "colour_heterogeneity", }  # Specify the pseudo-concepts to include
# concept_keys_to_show = {}  # Specify the pseudo-concepts to include

# save_path = FIG_DIR / f"{image_stem}_visual_grid.png" # Optional: specify a path to save the figure
save_path = ""  # Optional: specify a path to save the figure

show_visual_grid_for_stem(image_stem, xai_methods=xai_methods_to_show, concept_keys=concept_keys_to_show, save_path=save_path)


## 12. Export merged manifest for traceability

In [ ]:
resolved_path = METRICS_DIR / f"phase5_{RUN_NAME}_resolved_manifest.csv"
run_resolved.to_csv(resolved_path, index=False)

print("Saved resolved manifest:", resolved_path)
print("Done.")